In [1]:
# data from http://grouplens.org/datasets/movielens/

In [1]:
import os
#data_folder = os.path.join(os.path.expanduser("~"), "Data", "ml-100k")
ratings_filename = "C:/Users/xiao88160/Data/movie/ratings.csv"

In [2]:
import pandas as pd

In [3]:
all_ratings = pd.read_csv(ratings_filename)
all_ratings["timestamp"] = pd.to_datetime(all_ratings['timestamp'],unit='s')
all_ratings[:5]

,userId,movieId,rating,timestamp
0,1,31,2.5,2009-12-14 02:52:24
1,1,1029,3.0,2009-12-14 02:52:59
2,1,1061,3.0,2009-12-14 02:53:02
3,1,1129,2.0,2009-12-14 02:53:05
4,1,1172,4.0,2009-12-14 02:53:25


In [4]:
# As you can see, there are no review for most movies, such as #213
all_ratings[all_ratings["userId"] == 1].sort_values("movieId")  

,userId,movieId,rating,timestamp
0,1,31,2.5,2009-12-14 02:52:24
1,1,1029,3.0,2009-12-14 02:52:59
2,1,1061,3.0,2009-12-14 02:53:02
3,1,1129,2.0,2009-12-14 02:53:05
4,1,1172,4.0,2009-12-14 02:53:25
5,1,1263,2.0,2009-12-14 02:52:31
6,1,1287,2.0,2009-12-14 02:53:07
7,1,1293,2.0,2009-12-14 02:52:28
8,1,1339,3.5,2009-12-14 02:52:05
9,1,1343,2.0,2009-12-14 02:52:11


In [5]:
# Not all reviews are favourable! Our goal is "other recommended books", so we only want favourable reviews
all_ratings["Favorable"] = all_ratings["rating"] > 3
all_ratings[10:15]

,userId,movieId,rating,timestamp,Favorable
10,1,1371,2.5,2009-12-14 02:52:15,False
11,1,1405,1.0,2009-12-14 02:53:23,False
12,1,1953,4.0,2009-12-14 02:53:11,True
13,1,2105,4.0,2009-12-14 02:52:19,True
14,1,2150,3.0,2009-12-14 02:53:14,False


In [6]:
all_ratings[all_ratings["userId"] == 1][:5]

,userId,movieId,rating,timestamp,Favorable
0,1,31,2.5,2009-12-14 02:52:24,False
1,1,1029,3.0,2009-12-14 02:52:59,False
2,1,1061,3.0,2009-12-14 02:53:02,False
3,1,1129,2.0,2009-12-14 02:53:05,False
4,1,1172,4.0,2009-12-14 02:53:25,True


In [7]:
# Sample the dataset. You can try increasing the size of the sample, but the run time will be considerably longer
ratings = all_ratings[all_ratings['userId'].isin(range(200))]  # & ratings["UserID"].isin(range(100))]

In [8]:
# We start by creating a dataset of each user's favourable reviews
favorable_ratings = ratings[ratings["Favorable"]]
favorable_ratings[:5]

,userId,movieId,rating,timestamp,Favorable
4,1,1172,4.0,2009-12-14 02:53:25,True
8,1,1339,3.5,2009-12-14 02:52:05,True
12,1,1953,4.0,2009-12-14 02:53:11,True
13,1,2105,4.0,2009-12-14 02:52:19,True
20,2,10,4.0,1996-06-21 11:11:33,True


In [9]:
# We are only interested in the reviewers who have more than one review
favorable_reviews_by_users = dict((k, frozenset(v.values)) for k, v in favorable_ratings.groupby("userId")["movieId"])
len(favorable_reviews_by_users)

199

In [10]:
# Find out how many movies have favourable ratings
num_favorable_by_movie = ratings[["movieId", "Favorable"]].groupby("movieId").sum()
num_favorable_by_movie.sort_values("Favorable", ascending=False)[:5]

,Favorable
movieId,
296,88.0
318,84.0
356,84.0
593,72.0
527,70.0


In [11]:
from collections import defaultdict

def find_frequent_itemsets(favorable_reviews_by_users, k_1_itemsets, min_support):
    counts = defaultdict(int)
    for user, reviews in favorable_reviews_by_users.items():
        for itemset in k_1_itemsets:
            if itemset.issubset(reviews):
                for other_reviewed_movie in reviews - itemset:
                    current_superset = itemset | frozenset((other_reviewed_movie,))
                    counts[current_superset] += 1
    return dict([(itemset, frequency) for itemset, frequency in counts.items() if frequency >= min_support])

In [12]:
import sys
frequent_itemsets = {}  # itemsets are sorted by length
min_support = 50

# k=1 candidates are the isbns with more than min_support favourable reviews
frequent_itemsets[1] = dict((frozenset((movie_id,)), row["Favorable"])
                                for movie_id, row in num_favorable_by_movie.iterrows()
                                if row["Favorable"] > min_support)

print("There are {} movies with more than {} favorable reviews".format(len(frequent_itemsets[1]), min_support))
sys.stdout.flush()
for k in range(2, 20):
    # Generate candidates of length k, using the frequent itemsets of length k-1
    # Only store the frequent itemsets
    cur_frequent_itemsets = find_frequent_itemsets(favorable_reviews_by_users, frequent_itemsets[k-1],
                                                   min_support)
    if len(cur_frequent_itemsets) == 0:
        print("Did not find any frequent itemsets of length {}".format(k))
        sys.stdout.flush()
        break
    else:
        print("I found {} frequent itemsets of length {}".format(len(cur_frequent_itemsets), k))
        #print(cur_frequent_itemsets)
        sys.stdout.flush()
        frequent_itemsets[k] = cur_frequent_itemsets
# We aren't interested in the itemsets of length 1, so remove those
del frequent_itemsets[1]

There are 21 movies with more than 50 favorable reviews
I found 157 frequent itemsets of length 2
I found 590 frequent itemsets of length 3
I found 1250 frequent itemsets of length 4
I found 1596 frequent itemsets of length 5
I found 1279 frequent itemsets of length 6
I found 650 frequent itemsets of length 7
I found 199 frequent itemsets of length 8
I found 32 frequent itemsets of length 9
I found 2 frequent itemsets of length 10
Did not find any frequent itemsets of length 11


In [13]:
print("Found a total of {0} frequent itemsets".format(sum(len(itemsets) for itemsets in frequent_itemsets.values())))

Found a total of 5755 frequent itemsets


In [14]:
# Now we create the association rules. First, they are candidates until the confidence has been tested
candidate_rules = []
for itemset_length, itemset_counts in frequent_itemsets.items():
    for itemset in itemset_counts.keys():
        for conclusion in itemset:
            premise = itemset - set((conclusion,))
            candidate_rules.append((premise, conclusion))
print("There are {} candidate rules".format(len(candidate_rules)))

There are 29188 candidate rules


In [15]:
print(candidate_rules[:5])

[(frozenset({527}), 50), (frozenset({50}), 527), (frozenset({50}), 296), (frozenset({296}), 50), (frozenset({527}), 110)]


In [16]:
# Now, we compute the confidence of each of these rules. This is very similar to what we did in chapter 1
correct_counts = defaultdict(int)
incorrect_counts = defaultdict(int)
for user, reviews in favorable_reviews_by_users.items():
    for candidate_rule in candidate_rules:
        premise, conclusion = candidate_rule
        if premise.issubset(reviews):
            if conclusion in reviews:
                correct_counts[candidate_rule] += 1
            else:
                incorrect_counts[candidate_rule] += 1
rule_confidence = {candidate_rule: correct_counts[candidate_rule] / float(correct_counts[candidate_rule] + incorrect_counts[candidate_rule])
              for candidate_rule in candidate_rules}

In [17]:
# Choose only rules above a minimum confidence level
min_confidence = 0.9

In [18]:
# Filter out the rules with poor confidence
rule_confidence = {rule: confidence for rule, confidence in rule_confidence.items() if confidence > min_confidence}
print(len(rule_confidence))

9245


In [19]:
from operator import itemgetter
sorted_confidence = sorted(rule_confidence.items(), key=itemgetter(1), reverse=True)

In [20]:
for index in range(5):
    print("Rule #{0}".format(index + 1))
    (premise, conclusion) = sorted_confidence[index][0]
    print("Rule: If a person recommends {0} they will also recommend {1}".format(premise, conclusion))
    print(" - Confidence: {0:.3f}".format(rule_confidence[(premise, conclusion)]))
    print("")

Rule #1
Rule: If a person recommends frozenset({1210, 589}) they will also recommend 1196
 - Confidence: 1.000

Rule #2
Rule: If a person recommends frozenset({296, 1210, 589}) they will also recommend 1196
 - Confidence: 1.000

Rule #3
Rule: If a person recommends frozenset({296, 480, 1196}) they will also recommend 260
 - Confidence: 1.000

Rule #4
Rule: If a person recommends frozenset({296, 480, 1210}) they will also recommend 260
 - Confidence: 1.000

Rule #5
Rule: If a person recommends frozenset({296, 1210, 356}) they will also recommend 1196
 - Confidence: 1.000



In [21]:
# Even better, we can get the movie titles themselves from the dataset
#movie_name_filename = os.path.join(data_folder, "u.item")
movie_name_data = pd.read_csv('C:/Users/xiao88160/Data/movie/movies.csv')


In [22]:
def get_movie_name(movie_id):
    title_object = movie_name_data[movie_name_data["movieId"] == movie_id]["title"]
    title = title_object.values[0]
    return title

In [23]:
get_movie_name(4)

'Waiting to Exhale (1995)'

In [24]:
for index in range(5):
    print("Rule #{0}".format(index + 1))
    (premise, conclusion) = sorted_confidence[index][0]
    premise_names = ", ".join(get_movie_name(idx) for idx in premise)
    conclusion_name = get_movie_name(conclusion)
    print("Rule: If a person recommends {0} they will also recommend {1}".format(premise_names, conclusion_name))
    print(" - Confidence: {0:.3f}".format(rule_confidence[(premise, conclusion)]))
    print("")

Rule #1
Rule: If a person recommends Star Wars: Episode VI - Return of the Jedi (1983), Terminator 2: Judgment Day (1991) they will also recommend Star Wars: Episode V - The Empire Strikes Back (1980)
 - Confidence: 1.000

Rule #2
Rule: If a person recommends Pulp Fiction (1994), Star Wars: Episode VI - Return of the Jedi (1983), Terminator 2: Judgment Day (1991) they will also recommend Star Wars: Episode V - The Empire Strikes Back (1980)
 - Confidence: 1.000

Rule #3
Rule: If a person recommends Pulp Fiction (1994), Jurassic Park (1993), Star Wars: Episode V - The Empire Strikes Back (1980) they will also recommend Star Wars: Episode IV - A New Hope (1977)
 - Confidence: 1.000

Rule #4
Rule: If a person recommends Pulp Fiction (1994), Jurassic Park (1993), Star Wars: Episode VI - Return of the Jedi (1983) they will also recommend Star Wars: Episode IV - A New Hope (1977)
 - Confidence: 1.000

Rule #5
Rule: If a person recommends Pulp Fiction (1994), Star Wars: Episode VI - Return of

In [25]:
# Evaluation using test data
test_dataset = all_ratings[~all_ratings['userId'].isin(range(200))]
test_favorable = test_dataset[test_dataset["Favorable"]]
#test_not_favourable = test_dataset[~test_dataset["Favourable"]]
test_favorable_by_users = dict((k, frozenset(v.values)) for k, v in test_favorable.groupby("userId")["movieId"])
#test_not_favourable_by_users = dict((k, frozenset(v.values)) for k, v in test_not_favourable.groupby("UserID")["MovieID"])
#test_users = test_dataset["UserID"].unique()

In [26]:
test_dataset[:5]

,userId,movieId,rating,timestamp,Favorable
27425,200,1,3.0,2015-07-26 17:45:19,False
27426,200,2,3.5,2016-03-11 18:31:48,True
27427,200,32,4.0,2015-07-26 18:16:24,True
27428,200,110,3.5,2015-07-27 17:46:39,True
27429,200,145,4.5,2015-07-26 17:52:34,True


In [27]:
correct_counts = defaultdict(int)
incorrect_counts = defaultdict(int)
for user, reviews in test_favorable_by_users.items():
    for candidate_rule in candidate_rules:
        premise, conclusion = candidate_rule
        if premise.issubset(reviews):
            if conclusion in reviews:
                correct_counts[candidate_rule] += 1
            else:
                incorrect_counts[candidate_rule] += 1

In [28]:
test_confidence = {candidate_rule: correct_counts[candidate_rule] / float(correct_counts[candidate_rule] + incorrect_counts[candidate_rule])
                   for candidate_rule in rule_confidence}
print(len(test_confidence))

9245


In [29]:
sorted_test_confidence = sorted(test_confidence.items(), key=itemgetter(1), reverse=True)
print(sorted_test_confidence[:5])

[((frozenset({4993, 1196, 1198}), 260), 1.0), ((frozenset({457, 1210, 1198}), 260), 1.0), ((frozenset({4993, 2858, 1196}), 260), 1.0), ((frozenset({4993, 1210, 2858}), 260), 1.0), ((frozenset({4993, 1210, 2959}), 260), 1.0)]


In [30]:
for index in range(10):
    print("Rule #{0}".format(index + 1))
    (premise, conclusion) = sorted_confidence[index][0]
    premise_names = ", ".join(get_movie_name(idx) for idx in premise)
    conclusion_name = get_movie_name(conclusion)
    print("Rule: If a person recommends {0} they will also recommend {1}".format(premise_names, conclusion_name))
    print(" - Train Confidence: {0:.3f}".format(rule_confidence.get((premise, conclusion), -1)))
    print(" - Test Confidence: {0:.3f}".format(test_confidence.get((premise, conclusion), -1)))
    print("")

Rule #1
Rule: If a person recommends Star Wars: Episode VI - Return of the Jedi (1983), Terminator 2: Judgment Day (1991) they will also recommend Star Wars: Episode V - The Empire Strikes Back (1980)
 - Train Confidence: 1.000
 - Test Confidence: 0.903

Rule #2
Rule: If a person recommends Pulp Fiction (1994), Star Wars: Episode VI - Return of the Jedi (1983), Terminator 2: Judgment Day (1991) they will also recommend Star Wars: Episode V - The Empire Strikes Back (1980)
 - Train Confidence: 1.000
 - Test Confidence: 0.944

Rule #3
Rule: If a person recommends Pulp Fiction (1994), Jurassic Park (1993), Star Wars: Episode V - The Empire Strikes Back (1980) they will also recommend Star Wars: Episode IV - A New Hope (1977)
 - Train Confidence: 1.000
 - Test Confidence: 0.892

Rule #4
Rule: If a person recommends Pulp Fiction (1994), Jurassic Park (1993), Star Wars: Episode VI - Return of the Jedi (1983) they will also recommend Star Wars: Episode IV - A New Hope (1977)
 - Train Confiden